In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

# *Imports*

In [ ]:
import os
import re
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import wandb
seed = 42
random.seed(seed)
np.random.seed(seed)
sns.set_style('whitegrid')
print('Library loaded')


In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Kaggle Secret se key lo — automatically milegi
secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
print("✅ W&B login successful!")

# *Load Data*

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print(train_df.shape)
print(test_df.shape)

In [ ]:
train_df.head(5)


# *Null Check*

In [ ]:
print(train_df.info())
print(f"Null Values :\n{train_df.isnull().sum()}")
print(train_df['answer'].value_counts())

# *Answer Distribution*

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x=train_df['answer'])
plt.title('Answer Distribution')

# *Text Cleaning*

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]','',text)
    text = re.sub(r'\s+',' ',text)
    return text.strip()

In [ ]:
train_df['prompt'] = train_df['prompt'].apply(clean_text)
test_df['prompt'] = test_df['prompt'].apply(clean_text)
for col in ['A','B','C','D','E']:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)
print('Cleaning Text')

# *Convert Dataset for Classical ML*

In [ ]:
# rows = []
# for _,row in train_df.iterrows():
#     for option in ['A','B','C','D','E']:
#         rows.append({
#              'text': row['prompt'] + '[SEP]' + row[option],
#              'label': 1 if option == row['answer'] else 0 ,
#              'question_id': row['id'],
#              'option': option
        
#         })
        
       
# ml_df = pd.DataFrame(rows)
# print('Expanded Shape:',ml_df.shape)
# ml_df.head(5)
        

In [ ]:
rows = []

for _, row in train_df.iterrows():
    for option in ['A','B','C','D','E']:
        rows.append({
            'text': clean_text(
                f"Question: {row['prompt']} Answer: {row[option]}"
            ),
            'label': 1 if option == row['answer'] else 0,
            'question_id': row['id'],
            'option': option
        })

ml_df = pd.DataFrame(rows)

print(ml_df.head())

# *Train-Test Split*

In [ ]:
train_data , val_data = train_test_split(
    ml_df , test_size = 0.2 , random_state = seed , stratify = ml_df['label']
)
print(train_data.shape)
print(val_data.shape)

# *Tfidf*

In [ ]:
vectorizer = TfidfVectorizer(
    max_features = 60000,
    ngram_range = (1,2),
    stop_words = 'english',
    max_df = 0.95,
    min_df = 2,
    sublinear_tf = True
)
X_train = vectorizer.fit_transform(train_data['text'])
X_val = vectorizer.transform(val_data['text'])
y_train = train_data['label']
y_val = val_data['label']
print(X_train.shape)


# *Logistic Regression Model*

In [ ]:
lr_model = LogisticRegression(
    C=4,
    max_iter = 6000,
    class_weight = 'balanced',
    solver = 'liblinear',
    random_state = seed
)
lr_model.fit(X_train,y_train)
print('Logistic Regession Trained')

# *Validation Accuracy*

In [ ]:
from sklearn.metrics import classification_report
val_preds = lr_model.predict(X_val)
acc = accuracy_score(y_val,val_preds)
print('Validation Accuracy:',acc)
print(classification_report(y_val,val_preds))

# *MAP@3 Function*

In [ ]:
def map_at_3(actual,predicted):
    scores = []
    for true_ans , pred_list in zip(actual,predicted):
        score = 0
        for rank,pred in enumerate(pred_list[:3],start=1):
            if pred ==true_ans:
                score = 1/rank
                break
        scores.append(score)
    return np.mean(scores)
print('Done')

# *Logistic Regression Map@3*

In [ ]:
val_data['prob'] = lr_model.predict_proba(X_val)[:,1]

grouped = val_data.groupby('question_id')

predictions = []
actuals = []

for qid, group in grouped:
    ranked = group.sort_values('prob', ascending=False)['option'].tolist()

    predictions.append(ranked[:3])

    actual = train_df.loc[train_df['id']==qid, 'answer'].values[0]
    actuals.append(actual)

score = map_at_3(actuals, predictions)

print("Logistic Regression MAP@3:", score)

# *Naive Bayes*

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

val_data['nb_prob'] = nb_model.predict_proba(X_val)[:,1]

predictions_nb = []

for qid, group in val_data.groupby('question_id'):
    ranked = group.sort_values('nb_prob', ascending=False)['option'].tolist()
    predictions_nb.append(ranked[:3])

score_nb = map_at_3(actuals, predictions_nb)

print("Naive Bayes MAP@3:", score_nb)

# *Linear Svc*

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
model_Svc = LinearSVC(
    C=4.0,
    class_weight='balanced',
    max_iter=6000
)

model_Svc.fit(X_train, y_train)

val_pred = model_Svc.predict(X_val)

print(classification_report(y_val, val_pred))


In [ ]:
# LinearSVC mein predict_proba nahi hota
# decision_function use karo — higher score = zyada confident
val_data['svc_prob'] = model_Svc.decision_function(X_val)

predictions_svc = []
actuals_svc = []

for qid, group in val_data.groupby('question_id'):
    ranked = group.sort_values('svc_prob', ascending=False)['option'].tolist()
    predictions_svc.append(ranked[:3])

    actual = train_df.loc[train_df['id'] == qid, 'answer'].values[0]
    actuals_svc.append(actual)

score_svc = map_at_3(actuals_svc, predictions_svc)
print(f"LinearSVC MAP@3: {score_svc:.4f}")

# *LightGBM*

In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=seed,
    verbose=-1
)
lgbm_model.fit(X_train, y_train)

val_data['lgbm_prob'] = lgbm_model.predict_proba(X_val)[:, 1]

predictions_lgbm = []
for qid, group in val_data.groupby('question_id'):
    ranked = group.sort_values('lgbm_prob', ascending=False)['option'].tolist()
    predictions_lgbm.append(ranked[:3])

score_lgbm = map_at_3(actuals, predictions_lgbm)
print(f"LightGBM MAP@3: {score_lgbm:.4f}")

# *Ensemble*

In [ ]:
val_data['ensemble_prob'] = (
    val_data['prob'] * 0.4 +
    val_data['lgbm_prob'] * 0.6
)

predictions_ensemble = []
for qid, group in val_data.groupby('question_id'):
    ranked = group.sort_values('ensemble_prob', ascending=False)['option'].tolist()
    predictions_ensemble.append(ranked[:3])

score_ensemble = map_at_3(actuals, predictions_ensemble)
print(f"Ensemble MAP@3: {score_ensemble:.4f}")

# *Compare MOdels*

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'LinearSVC','Naive Bayes','LightGBM','Ensemble'],
    'MAP@3': [score, score_nb,score_svc,score_lgbm,score_ensemble]
})

results


# *Plot Results*

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=results, x='Model', y='MAP@3')
plt.axhline(0.85, linestyle='--')
plt.title("Classical ML Comparison")
plt.show()

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# API key Kaggle Secret se lo
secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

# W&B run start karo
wandb.init(
    project="23f2005181-t22026",
    name="classical-ml",
    config={
        "vectorizer": "TF-IDF",
        "train_size": len(train_data),
        "val_size": len(val_data)
    }
)

wandb.log({
    "logistic_map3": score,
    "linear_svc_mp@3":score_svc,
    "naive_bayes_map3": score_nb,
    "lgbm_map3": score_lgbm,
    "ensemble_map3": score_ensemble
})
wandb.finish()
print(" W&B done!")

In [ ]:
# Step 1 — Test data expand karo (bilkul train jaisa)
test_rows = []
for _, row in test_df.iterrows():
    for option in ['A', 'B', 'C', 'D', 'E']:
        test_rows.append({
            'text': clean_text(f"Question: {row['prompt']} Answer: {row[option]}"),
           
            'question_id': row['id'],
            'option': option
        })

test_expanded = pd.DataFrame(test_rows)
print('Test expanded shape:', test_expanded.shape)  # 2500 rows honi chahiye

# Step 2 — Vectorize (fit nahi, sirf transform)
X_test_vec = vectorizer.transform(test_expanded['text'])

# Step 3 — Probability lo (class 1 = correct hone ki probability)
test_expanded['prob'] = lr_model.predict_proba(X_test_vec)[:, 1]

# Step 4 — Har question ke liye top 3 rank karo
final_predictions = []
for qid, group in test_expanded.groupby('question_id'):
    ranked = group.sort_values('prob', ascending=False)['option'].tolist()
    final_predictions.append(' '.join(ranked[:3]))

# Step 5 — Submission banao
submission = pd.DataFrame({
    'ID': test_df['id'],
    'Prediction': final_predictions
})

# Format check
assert len(submission) == 500, '500 rows honi chahiye!'
assert all(len(p.split()) == 3 for p in submission['Prediction']), 'Har row mein 3 options chahiye!'
print('Format check passed!')
print(submission.head(10))

submission.to_csv('submission.csv', index=False)
print('submission.csv ready!')